# Notebook 1: Crystal Structures & Materials Data

**APS Tutorial T4 — Generative AI for Physics: From Models to Materials**

In this notebook you will learn to:
- Represent crystal structures in Python using **pymatgen**
- Create structures from scratch (lattice + basis)
- Load structures from CIF files
- Inspect lattice parameters, compositions, and coordinates
- Visualize unit cells in 3D
- Explore the **MP-20 training dataset** (property distributions, element frequencies)
- Understand why generative models are needed for exotic lattice geometries

> **Prerequisites:** Run `00_setup.ipynb` first to install dependencies.

In [ ]:
# --- Run once per Colab session (skip if you already ran 00_setup.ipynb) ---
import os, shutil, subprocess, sys

REPO_URL = 'https://github.com/RyotaroOKabe/APS_demo_SCIGEN.git'
PROJECT_DIR = '/content/APS_demo_SCIGEN'

# Clone repo if needed
if not os.path.exists(os.path.join(PROJECT_DIR, '.git')):
    if os.path.exists(PROJECT_DIR):
        shutil.rmtree(PROJECT_DIR)
    os.system(f'git clone {REPO_URL} {PROJECT_DIR}')

# Install all tutorial dependencies from the shared requirements file
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       '-r', f'{PROJECT_DIR}/requirements-colab.txt'])
os.environ.setdefault('PROJECT_ROOT', PROJECT_DIR)
print('Ready.')

---
## 1.1 What is a crystal structure?

A crystal structure is defined by:
- A **lattice** — the periodic box described by three vectors **a**, **b**, **c** (or equivalently, lengths *a, b, c* and angles *α, β, γ*)
- A **basis** — the atoms inside one unit cell, given by their species and fractional coordinates

In pymatgen, these are combined into a `Structure` object.

### Fractional vs. Cartesian coordinates

Atoms in a crystal can be described two ways:
- **Cartesian coordinates** (x, y, z in Angstroms) — absolute positions in space
- **Fractional coordinates** (a, b, c in [0, 1)) — positions relative to the lattice vectors

Fractional coordinates are more natural for crystals because they are independent of the unit cell size. An atom at fractional coordinate (0.5, 0.5, 0.5) is always at the center of the cell, regardless of how large or small that cell is.

In [ ]:
from pymatgen.core import Structure, Lattice
import numpy as np

## 1.2 Creating a structure from scratch

Let's build a simple cubic NaCl (rock salt) structure.
The conventional cell has lattice parameter *a* ≈ 5.64 Å.

In [ ]:
# Define the lattice: cubic with a = 5.64 Å
lattice = Lattice.cubic(5.64)

# Define species and fractional coordinates
species = ['Na', 'Na', 'Na', 'Na', 'Cl', 'Cl', 'Cl', 'Cl']
frac_coords = [
    [0.0, 0.0, 0.0],  # Na
    [0.5, 0.5, 0.0],  # Na
    [0.5, 0.0, 0.5],  # Na
    [0.0, 0.5, 0.5],  # Na
    [0.5, 0.0, 0.0],  # Cl
    [0.0, 0.5, 0.0],  # Cl
    [0.0, 0.0, 0.5],  # Cl
    [0.5, 0.5, 0.5],  # Cl
]

nacl = Structure(lattice, species, frac_coords)
print(nacl)

### Inspecting a structure

pymatgen structures expose all the information you need:

In [ ]:
print(f'Formula:           {nacl.composition.reduced_formula}')
print(f'Number of atoms:   {nacl.num_sites}')
print(f'Volume:            {nacl.volume:.2f} Å³')
print(f'Density:           {nacl.density:.3f} g/cm³')
print()
print('Lattice parameters:')
print(f'  a={nacl.lattice.a:.2f} Å, b={nacl.lattice.b:.2f} Å, c={nacl.lattice.c:.2f} Å')
print(f'  α={nacl.lattice.alpha:.1f}°, β={nacl.lattice.beta:.1f}°, γ={nacl.lattice.gamma:.1f}°')
print()
print('Fractional coordinates:')
for site in nacl:
    print(f'  {site.species_string:>4s}  {site.frac_coords}')

---
## 1.3 Loading structures & exploring the MP-20 dataset

CIF (Crystallographic Information File) is the standard format for crystal structures.
SCIGEN was trained on the **MP-20** subset of the Materials Project: all structures
with 20 or fewer atoms per unit cell. The data is included as CSV files with CIF strings.

In [ ]:
import pandas as pd
import os

# Load the SCIGEN training data (MP-20 dataset)
PROJECT_DIR = os.environ.get('PROJECT_ROOT', '/content/APS_demo_SCIGEN')
df = pd.read_csv(os.path.join(PROJECT_DIR, 'data', 'mp_20', 'train.csv'))
print(f'Dataset: {len(df)} structures')
print(f'Columns: {list(df.columns)}')

In [ ]:
# Look at the first few entries
df[['material_id', 'pretty_formula', 'formation_energy_per_atom',
    'e_above_hull', 'spacegroup.number']].head(10)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Formation energy distribution
axes[0].hist(df['formation_energy_per_atom'], bins=60, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Formation energy (eV/atom)')
axes[0].set_ylabel('Count')
axes[0].set_title('Formation energy distribution')

# Band gap distribution
axes[1].hist(df['band_gap'], bins=60, color='coral', edgecolor='white')
axes[1].set_xlabel('Band gap (eV)')
axes[1].set_ylabel('Count')
axes[1].set_title('Band gap distribution')

# Energy above hull
axes[2].hist(df['e_above_hull'], bins=60, color='seagreen', edgecolor='white',
             range=(0, 0.5))
axes[2].set_xlabel('Energy above hull (eV/atom)')
axes[2].set_ylabel('Count')
axes[2].set_title('Stability (e_above_hull)')

plt.tight_layout()
plt.show()

In [ ]:
# Periodic table heatmap of element frequency in training data
from collections import Counter
import ast
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

all_elements = []
for elem_str in df['elements']:
    try:
        elems = ast.literal_eval(elem_str)
        all_elements.extend(elems)
    except:
        pass

elem_counts = Counter(all_elements)

# Periodic table layout (row, col) for each element
pt_layout = {
    'H': (0,0), 'He': (0,17),
    'Li': (1,0), 'Be': (1,1), 'B': (1,12), 'C': (1,13), 'N': (1,14), 'O': (1,15), 'F': (1,16), 'Ne': (1,17),
    'Na': (2,0), 'Mg': (2,1), 'Al': (2,12), 'Si': (2,13), 'P': (2,14), 'S': (2,15), 'Cl': (2,16), 'Ar': (2,17),
    'K': (3,0), 'Ca': (3,1), 'Sc': (3,2), 'Ti': (3,3), 'V': (3,4), 'Cr': (3,5), 'Mn': (3,6), 'Fe': (3,7),
    'Co': (3,8), 'Ni': (3,9), 'Cu': (3,10), 'Zn': (3,11), 'Ga': (3,12), 'Ge': (3,13), 'As': (3,14),
    'Se': (3,15), 'Br': (3,16), 'Kr': (3,17),
    'Rb': (4,0), 'Sr': (4,1), 'Y': (4,2), 'Zr': (4,3), 'Nb': (4,4), 'Mo': (4,5), 'Tc': (4,6), 'Ru': (4,7),
    'Rh': (4,8), 'Pd': (4,9), 'Ag': (4,10), 'Cd': (4,11), 'In': (4,12), 'Sn': (4,13), 'Sb': (4,14),
    'Te': (4,15), 'I': (4,16), 'Xe': (4,17),
    'Cs': (5,0), 'Ba': (5,1), 'La': (5,2), 'Hf': (5,3), 'Ta': (5,4), 'W': (5,5), 'Re': (5,6), 'Os': (5,7),
    'Ir': (5,8), 'Pt': (5,9), 'Au': (5,10), 'Hg': (5,11), 'Tl': (5,12), 'Pb': (5,13), 'Bi': (5,14),
}

fig, ax = plt.subplots(figsize=(14, 6))
max_count = max(elem_counts.values()) if elem_counts else 1
cmap = plt.cm.YlOrRd

for el, (row, col) in pt_layout.items():
    count = elem_counts.get(el, 0)
    color = cmap(count / max_count) if count > 0 else '#f0f0f0'
    rect = mpatches.FancyBboxPatch((col, -row), 0.9, 0.9, boxstyle='round,pad=0.05',
                                     facecolor=color, edgecolor='gray', linewidth=0.5)
    ax.add_patch(rect)
    ax.text(col + 0.45, -row + 0.55, el, ha='center', va='center', fontsize=8, fontweight='bold')
    if count > 0:
        ax.text(col + 0.45, -row + 0.2, str(count), ha='center', va='center', fontsize=6, color='gray')

ax.set_xlim(-0.5, 18.5)
ax.set_ylim(-6.5, 1.5)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Element frequency in MP-20 training data', fontsize=14, pad=15)

# Colorbar
sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(0, max_count))
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, shrink=0.5, aspect=20, pad=0.02)
cbar.set_label('Number of structures', fontsize=10)

plt.tight_layout()
plt.show()

### Spacegroup distribution

Crystal structures are classified by their **space group** — the set of symmetry operations
(rotations, reflections, translations) that leave the structure unchanged. There are 230 possible
space groups. Let's see which ones dominate the training data:

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
sg_counts = df['spacegroup.number'].value_counts().head(15)
ax.bar(range(len(sg_counts)), sg_counts.values, color='steelblue', edgecolor='white')
ax.set_xticks(range(len(sg_counts)))
ax.set_xticklabels([f'SG {sg}' for sg in sg_counts.index], rotation=45, ha='right')
ax.set_ylabel('Count')
ax.set_title('Top 15 space groups in MP-20 training data')
plt.tight_layout()
plt.show()

### Unit cell sizes

SCIGEN is trained on structures with ≤20 atoms per unit cell (the "MP-20" subset).
Here's the distribution:

In [ ]:
# Count atoms per structure from CIF data
from pymatgen.core import Structure

n_atoms_list = []
for i in range(min(500, len(df))):  # sample for speed
    try:
        s = Structure.from_str(df.iloc[i]['cif'], fmt='cif')
        n_atoms_list.append(s.num_sites)
    except:
        pass

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(n_atoms_list, bins=range(1, 22), color='steelblue', edgecolor='white', align='left')
ax.set_xlabel('Atoms per unit cell')
ax.set_ylabel('Count (sample of 500)')
ax.set_title('Unit cell size distribution in MP-20')
ax.set_xticks(range(1, 21))
plt.tight_layout()
plt.show()

---
### How rare are kagome materials?

This is the motivating question for SCIGEN: if we search the known databases,
how many materials with a kagome sublattice do we find?

The answer is: **very few.** Kagome magnets are rare in nature, but they are
extremely interesting for physics (frustrated magnetism, spin liquids, flat bands).

SCIGEN's goal is to **generate new candidate materials** with these special lattice
geometries — expanding the search space far beyond what's been experimentally observed.

In [ ]:
# Count Mn-containing materials in the training data
mn_count = 0
for elem_str in df['elements']:
    try:
        if 'Mn' in ast.literal_eval(elem_str):
            mn_count += 1
    except:
        pass

print(f'Mn-containing structures in training data: {mn_count} / {len(df)}')
print(f'That\'s {mn_count/len(df)*100:.1f}% of the dataset.')
print()
print('Of these, even fewer have a kagome sublattice.')
print('This is exactly why we need generative models —')
print('to explore regions of materials space that are underrepresented in databases.')

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.colors import to_rgba

# CPK-inspired colorblind-safe palette
ELEMENT_COLORS = {
    'H': '#FFFFFF', 'Li': '#CC80FF', 'Be': '#C2FF00', 'B': '#FFB5B5',
    'C': '#404040', 'N': '#3050F8', 'O': '#FF0D0D', 'F': '#90E050',
    'Na': '#AB5CF2', 'Mg': '#8AFF00', 'Al': '#BFA6A6', 'Si': '#F0C8A0',
    'P': '#FF8000', 'S': '#FFFF30', 'Cl': '#1FF01F', 'K': '#8F40D4',
    'Ca': '#3DFF00', 'Ti': '#BFC2C7', 'V': '#A6A6AB', 'Cr': '#8A99C7',
    'Mn': '#9C7AC7', 'Fe': '#E06633', 'Co': '#F090A0', 'Ni': '#50D050',
    'Cu': '#C88033', 'Zn': '#7D80B0', 'Ga': '#C28F8F', 'Ge': '#668F8F',
    'As': '#BD80E3', 'Se': '#FFA100', 'Br': '#A62929', 'Sr': '#00FF00',
    'Y': '#94FFFF', 'Zr': '#94E0E0', 'Nb': '#73C2C9', 'Mo': '#54B5B5',
    'Ru': '#248F8F', 'Rh': '#0A7D8C', 'Pd': '#006985', 'Ag': '#C0C0C0',
    'In': '#A67573', 'Sn': '#668080', 'Sb': '#9E63B5', 'Te': '#D47A00',
    'I': '#940094', 'Ba': '#00C900', 'La': '#70D4FF', 'Ce': '#FFFFC7',
    'Bi': '#9E4FB5', 'Pb': '#575961',
}

_tab10 = plt.cm.tab10

def _get_element_color(symbol, _cache={}):
    """Return a color for the given element symbol."""
    if symbol in ELEMENT_COLORS:
        return ELEMENT_COLORS[symbol]
    if symbol not in _cache:
        idx = len(_cache) % 10
        _cache[symbol] = _tab10(idx)
    return _cache[symbol]

def plot_structure(struct, title=None, ax=None, bond_cutoff=None):
    """Plot a pymatgen Structure in 3D with bonds."""
    if ax is None:
        fig = plt.figure(figsize=(6, 6))
        ax = fig.add_subplot(111, projection='3d')

    # Draw unit cell edges first (thin, light, dashed)
    lattice_matrix = struct.lattice.matrix
    origin = np.array([0, 0, 0])
    a, b, c = lattice_matrix
    edges = [
        (origin, a), (origin, b), (origin, c),
        (a, a+b), (a, a+c), (b, a+b), (b, b+c),
        (c, a+c), (c, b+c), (a+b, a+b+c), (a+c, a+b+c), (b+c, a+b+c)
    ]
    for start, end in edges:
        ax.plot(*zip(start, end), color='#cccccc', linestyle='--',
                linewidth=0.7, alpha=0.6)

    # Draw bonds (solid, colored, thicker — clearly distinct from cell edges)
    if bond_cutoff is None:
        bond_cutoff = 3.0
    all_neighbors = struct.get_all_neighbors(bond_cutoff)
    drawn = set()
    for i, neighbors in enumerate(all_neighbors):
        for neighbor in neighbors:
            j = neighbor.index
            pair = (min(i, j), max(i, j))
            if pair not in drawn:
                drawn.add(pair)
                p1 = struct[i].coords
                p2 = neighbor.coords
                ax.plot([p1[0], p2[0]], [p1[1], p2[1]], [p1[2], p2[2]],
                        color='#2196F3', linewidth=1.8, alpha=0.7, solid_capstyle='round')

    # Plot atoms on top
    cart_coords = struct.cart_coords
    for site in struct:
        c = _get_element_color(site.species_string)
        ax.scatter(*site.coords, s=200, c=[c], edgecolors='black', linewidths=0.5,
                   depthshade=True, label=site.species_string, zorder=5)

    # De-duplicate legend entries
    handles, labels = ax.get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    ax.legend(by_label.values(), by_label.keys(), loc='upper left', fontsize=9)

    if title:
        ax.set_title(title, fontsize=12)
    ax.set_xlabel('x (Å)')
    ax.set_ylabel('y (Å)')
    ax.set_zlabel('z (Å)')
    return ax

### Browse the dataset interactively

Use the slider below to explore different structures in the training data:

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    out = widgets.Output()

    def show_structure(idx):
        with out:
            clear_output(wait=True)
            row = df.iloc[idx]
            s = Structure.from_str(row['cif'], fmt='cif')
            print(f'#{idx}: {row["pretty_formula"]}  |  {row["material_id"]}  |  '
                  f'{s.num_sites} atoms  |  SG {row["spacegroup.number"]}  |  '
                  f'Ef = {row["formation_energy_per_atom"]:.3f} eV/atom')
            fig = plt.figure(figsize=(5, 5))
            ax = fig.add_subplot(111, projection='3d')
            plot_structure(s, title=row['pretty_formula'], ax=ax)
            plt.tight_layout()
            plt.show()

    slider = widgets.IntSlider(min=0, max=len(df)-1, step=1, value=0, description='Index:')
    widgets.interact(show_structure, idx=slider)
    display(out)
except ImportError:
    print('ipywidgets not available — skipping interactive browser.')

In [ ]:
# Visualize NaCl
plot_structure(nacl, title='NaCl (rock salt)')
plt.tight_layout()
plt.show()

In [ ]:
# Save to CIF
cif_string = struct.to(fmt='cif')
print(cif_string[:500])

---
## 1.4 Visualizing crystal structures

Let's visualize a unit cell using matplotlib. We'll plot atoms as spheres
colored by element, and draw the unit cell box.

### Why are kagome materials interesting?

The kagome lattice creates **geometric frustration** in magnetic materials. On a triangle of antiferromagnetically coupled spins, it's impossible for all three spins to be antiparallel to each other simultaneously. This frustration suppresses conventional magnetic order and can give rise to:

- **Quantum spin liquids** — exotic states with long-range entanglement but no magnetic order
- **Flat electronic bands** — leading to strongly correlated electron physics
- **Anomalous Hall effects** — useful for spintronic devices

Despite their scientific importance, kagome materials are **rare in nature**. This is exactly the gap that SCIGEN aims to fill.

In [ ]:
# Visualize the structure we loaded from the dataset
plot_structure(struct, title=f'{row["pretty_formula"]} ({row["material_id"]})')
plt.tight_layout()
plt.show()

In [ ]:
# PyVista rendering of key structures from the dataset
try:
    from crystal_viz import plot_crystal, plot_crystal_grid

    # Render a grid: NaCl + dataset structure + kagome
    grid_structs = [nacl, struct]
    grid_titles = ['NaCl (rock salt)', f'{row["pretty_formula"]} ({row["material_id"]})']

    # Add a few more dataset structures for variety
    for idx in [5, 10, 15]:
        if idx < len(df):
            r = df.iloc[idx]
            try:
                s = Structure.from_str(r['cif'], fmt='cif')
                grid_structs.append(s)
                grid_titles.append(f'{r["pretty_formula"]} ({r["material_id"]})')
            except Exception:
                pass

    grid_img = plot_crystal_grid(grid_structs, titles=grid_titles,
                                  ncols=min(4, len(grid_structs)),
                                  atom_scale=0.3, elevation=65)
except ImportError:
    print('PyVista not available — skipping 3D grid rendering.')
except Exception as e:
    print(f'Grid rendering failed: {e}')

---
## Exercise

1. Pick a different structure from the training data (`df.iloc[100]`, for example). Parse it, print the formula and number of atoms, and visualize it.
2. Try building a **honeycomb** lattice (2 atoms per cell at [1/3, 2/3, 0] and [2/3, 1/3, 0] in a hexagonal cell). Visualize it as a supercell.
3. Create a 2x2x1 supercell of the kagome structure. How does the bond pattern look different from the single cell?

In [ ]:
# Show periodicity: 2x2x2 supercell of NaCl
supercell_nacl = nacl.copy()
supercell_nacl.make_supercell([2, 2, 2])

print(f'Unit cell: {nacl.num_sites} atoms')
print(f'2x2x2 supercell: {supercell_nacl.num_sites} atoms')

fig, axes = plt.subplots(1, 2, figsize=(12, 5), subplot_kw={'projection': '3d'})
plot_structure(nacl, title='Unit cell (8 atoms)', ax=axes[0])
plot_structure(supercell_nacl, title='2\u00d72\u00d72 supercell (64 atoms)', ax=axes[1])
plt.tight_layout()
plt.show()

---
## 1.5 The kagome lattice — our running example

The **kagome lattice** is a pattern of corner-sharing triangles that appears in many
frustrated magnets. It is one of the key structural constraints that SCIGEN can target.

Let's build a simple kagome lattice to see what it looks like.

In [ ]:
# Build a simple kagome lattice with Mn atoms
# Hexagonal cell: a = b, gamma = 120°
a_lat = 5.0  # Å
lattice_kag = Lattice.hexagonal(a_lat, 6.0)  # a=b=5.0, c=6.0, gamma=120°

# Kagome sites: 3 atoms per unit cell at specific fractional positions
kagome_frac = [
    [0.5, 0.0, 0.0],
    [0.0, 0.5, 0.0],
    [0.5, 0.5, 0.0],
]

kagome_struct = Structure(lattice_kag, ['Mn'] * 3, kagome_frac)

print('Kagome lattice with Mn:')
print(f'  a = {lattice_kag.a:.2f} Å, c = {lattice_kag.c:.2f} Å, γ = {lattice_kag.gamma:.0f}°')
print(f'  Atoms per cell: {kagome_struct.num_sites}')
print()
print(kagome_struct)

In [ ]:
# Visualize the kagome lattice
# We'll plot a 2x2 supercell in 2D to see the triangular pattern
import matplotlib.pyplot as plt
import numpy as np
from pymatgen.core import Structure, Lattice

# Ensure kagome_struct exists (recreate if cell above was skipped)
if 'kagome_struct' not in dir():
    a_lat = 5.0
    lattice_kag = Lattice.hexagonal(a_lat, 6.0)
    kagome_frac = [[0.5, 0.0, 0.0], [0.0, 0.5, 0.0], [0.5, 0.5, 0.0]]
    kagome_struct = Structure(lattice_kag, ['Mn'] * 3, kagome_frac)
else:
    a_lat = kagome_struct.lattice.a

fig, ax = plt.subplots(figsize=(7, 6))

# Create a 3x3 supercell to see the kagome pattern clearly
supercell = kagome_struct.copy()
supercell.make_supercell([3, 3, 1])

# Plot atom positions (projected onto xy plane)
coords = supercell.cart_coords
ax.scatter(coords[:, 0], coords[:, 1], s=120, c='orchid',
           edgecolors='black', linewidths=0.8, zorder=5)

# Draw bonds (connect atoms within ~0.6*a of each other)
from scipy.spatial.distance import cdist
dists = cdist(coords[:, :2], coords[:, :2])
bond_cutoff = a_lat * 0.6
for i in range(len(coords)):
    for j in range(i+1, len(coords)):
        if dists[i, j] < bond_cutoff:
            ax.plot([coords[i, 0], coords[j, 0]],
                    [coords[i, 1], coords[j, 1]],
                    'k-', linewidth=1.2, alpha=0.5)

ax.set_aspect('equal')
ax.set_title('Kagome lattice — corner-sharing triangles', fontsize=13)
ax.set_xlabel('x (Å)')
ax.set_ylabel('y (Å)')
plt.tight_layout()
plt.show()

The kagome pattern of corner-sharing triangles leads to **geometric frustration**
in magnetic materials — spins on a triangle cannot all be antiparallel to each other.
This frustration gives rise to exotic quantum states like spin liquids.

In Notebook 04, we will use SCIGEN to **generate new crystal structures** that
contain this kagome sublattice.

---
## Exercise

1. Pick a different structure from the training data (`df.iloc[100]`, for example). Parse it, print the formula and number of atoms, and visualize it.
2. Try building a **honeycomb** lattice (2 atoms per cell at [1/3, 2/3, 0] and [2/3, 1/3, 0] in a hexagonal cell). Visualize it as a supercell.

---
## 1.6 Tight-binding band structures: from geometry to electronics

Why do physicists care about specific lattice geometries? Because the **lattice shape
directly determines the electronic band structure**. Using a simple tight-binding (TB)
model, we can see how kagome, honeycomb, and triangular lattices produce qualitatively
different electronic properties.

Key predictions from tight-binding:
- **Kagome lattice** → **flat band** (localized electrons, strong correlations) + Dirac cone
- **Honeycomb lattice** → **Dirac cones** (graphene-like massless fermions)
- **Triangular lattice** → simple dispersive bands (no exotic features)

These electronic signatures are why SCIGEN targets specific sublattice constraints.

In [ ]:
try:
    from pythtb import TBModel, Lattice as TBLattice
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pythtb'])
    from pythtb import TBModel, Lattice as TBLattice

import matplotlib.pyplot as plt
import numpy as np

sqrt3 = np.sqrt(3.0)

# Hexagonal lattice vectors: a1 = a*(1, 0), a2 = a*(1/2, sqrt3/2)
# We set a = 1 (dimensionless); all energies in units of |t|
lat_vecs = [[1.0, 0.0], [0.5, sqrt3 / 2.0]]

# Standard high-symmetry path for hexagonal BZ: Gamma -> M -> K -> Gamma
k_nodes = [[0.0, 0.0], [0.5, 0.0], [1./3., 1./3.], [0.0, 0.0]]
k_labels = [r'$\Gamma$', r'$M$', r'$K$', r'$\Gamma$']
nk = 301

def make_model(lat_vecs, orb_vecs):
    """Create a 2D periodic TBModel with the given lattice and orbitals."""
    lat = TBLattice(lat_vecs=lat_vecs, orb_vecs=orb_vecs, periodic_dirs=[0, 1])
    return TBModel(lattice=lat)

# Use positive hopping t > 0 (standard convention in condensed matter)
t = 1.0

# ========================================================================
# (a) KAGOME LATTICE — 3 orbitals per cell
#     Sites at edge midpoints: NN distance = a/2
#     Expected: flat band at E = -2t, Dirac crossing, bandwidth = 6t
# ========================================================================
orb_kag = [[0.5, 0.0], [0.0, 0.5], [0.5, 0.5]]
kag = make_model(lat_vecs, orb_kag)
# 3 intra-cell + 3 inter-cell NN hoppings (each site has 4 NN)
kag.set_hop(t, 0, 1, [0, 0])
kag.set_hop(t, 0, 2, [0, 0])
kag.set_hop(t, 1, 2, [0, 0])
kag.set_hop(t, 0, 2, [0, -1])
kag.set_hop(t, 0, 1, [1, -1])
kag.set_hop(t, 1, 2, [-1, 0])

k_vec_k, k_dist_k, k_node_k = kag.k_path(k_nodes, nk, report=False)
evals_kag = kag.solve_ham(k_vec_k)

# ========================================================================
# (b) HONEYCOMB LATTICE — 2 orbitals per cell (graphene)
#     Standard sublattice positions: A at (0,0), B at (1/3, 1/3)
#     NN distance = a/sqrt(3), each atom has 3 NN
#     Expected: Dirac cone at K with E = 0, bandwidth = 6t
# ========================================================================
orb_hon = [[0.0, 0.0], [1./3., 1./3.]]
hon = make_model(lat_vecs, orb_hon)
hon.set_hop(t, 0, 1, [0, 0])    # B in same cell
hon.set_hop(t, 0, 1, [-1, 0])   # B in cell shifted by -a1
hon.set_hop(t, 0, 1, [0, -1])   # B in cell shifted by -a2

k_vec_h, k_dist_h, k_node_h = hon.k_path(k_nodes, nk, report=False)
evals_hon = hon.solve_ham(k_vec_h)

# ========================================================================
# (c) TRIANGULAR LATTICE — 1 orbital per cell
#     Single site at origin, 6 NN at distance a
#     Expected: cosine band, E(Gamma) = 6t, E(K) = 0, E(M) = -2t
# ========================================================================
orb_tri = [[0.0, 0.0]]
tri = make_model(lat_vecs, orb_tri)
tri.set_hop(t, 0, 0, [1, 0])    # +a1 direction
tri.set_hop(t, 0, 0, [0, 1])    # +a2 direction
tri.set_hop(t, 0, 0, [1, -1])   # +a1 - a2 direction

k_vec_t, k_dist_t, k_node_t = tri.k_path(k_nodes, nk, report=False)
evals_tri = tri.solve_ham(k_vec_t)

# ========================================================================
# Plot all three band structures
# ========================================================================
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

datasets = [
    (axes[0], evals_kag, k_dist_k, k_node_k, 'Kagome', '#9C7AC7'),
    (axes[1], evals_hon, k_dist_h, k_node_h, 'Honeycomb', '#3cb44b'),
    (axes[2], evals_tri, k_dist_t, k_node_t, 'Triangular', '#4363d8'),
]

for ax, evals, k_dist, k_node, title, color in datasets:
    for band_idx in range(evals.shape[1]):
        ax.plot(k_dist, evals[:, band_idx], color=color, linewidth=2.0)
    for node in k_node:
        ax.axvline(node, color='gray', linewidth=0.5, linestyle='--')
    ax.axhline(0, color='black', linewidth=0.3, alpha=0.3)
    ax.set_xticks(k_node)
    ax.set_xticklabels(k_labels, fontsize=12)
    ax.set_xlim(k_dist[0], k_dist[-1])
    ax.set_ylabel('E / t', fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.1)

# --- Kagome annotations ---
# Flat band at E = -2t = -2
flat_y = evals_kag[0, 0]  # lowest band (flat)
axes[0].axhline(flat_y, color='red', linewidth=0.8, linestyle=':', alpha=0.5)
axes[0].annotate('Flat band\n(E = $-$2t)',
                 xy=(k_dist_k[nk*3//4], flat_y),
                 xytext=(k_dist_k[nk*3//4] + 0.15, flat_y + 1.0),
                 arrowprops=dict(arrowstyle='->', color='red', lw=1.5),
                 fontsize=10, color='red', fontweight='bold')
# Dirac point
axes[0].annotate('Dirac\npoint',
                 xy=(k_node_k[2], evals_kag[np.argmin(np.abs(k_dist_k - k_node_k[2])), 1]),
                 xytext=(k_node_k[2] - 0.35, 0.0),
                 arrowprops=dict(arrowstyle='->', color='darkred', lw=1.2),
                 fontsize=9, color='darkred')

# --- Honeycomb annotations ---
# Dirac cone at K with E = 0
k_K_idx = np.argmin(np.abs(k_dist_h - k_node_h[2]))
axes[1].annotate('Dirac cone\n(E = 0 at K)',
                 xy=(k_dist_h[k_K_idx], 0),
                 xytext=(k_dist_h[k_K_idx] - 0.55, -1.5),
                 arrowprops=dict(arrowstyle='->', color='red', lw=1.5),
                 fontsize=10, color='red', fontweight='bold')

# --- Triangular annotations ---
axes[2].annotate('6t',
                 xy=(k_node_t[0] + 0.02, evals_tri[0, 0]),
                 fontsize=10, color='#4363d8', fontweight='bold', va='bottom')
axes[2].annotate('$-$2t',
                 xy=(k_node_t[1] + 0.02, evals_tri[np.argmin(np.abs(k_dist_t - k_node_t[1])), 0]),
                 fontsize=10, color='#4363d8', fontweight='bold', va='top')

plt.tight_layout()
plt.show()

# --- Verification table ---
print('=== Eigenvalues at high-symmetry points (units of t) ===')
print(f'{"Lattice":<12} {"Gamma":<20} {"M":<20} {"K":<20}')
print('-' * 72)
for name, evals, k_dist, k_node in [
    ('Kagome', evals_kag, k_dist_k, k_node_k),
    ('Honeycomb', evals_hon, k_dist_h, k_node_h),
    ('Triangular', evals_tri, k_dist_t, k_node_t),
]:
    iG = np.argmin(np.abs(k_dist - k_node[0]))
    iM = np.argmin(np.abs(k_dist - k_node[1]))
    iK = np.argmin(np.abs(k_dist - k_node[2]))
    def fmt(arr): return '[' + ', '.join(f'{v:+.1f}' for v in sorted(arr)) + ']'
    print(f'{name:<12} {fmt(evals[iG]):<20} {fmt(evals[iM]):<20} {fmt(evals[iK]):<20}')

print()
print('Expected (analytical):')
print(f'{"Kagome":<12} {"[-2, -2, +4]":<20} {"[-2,  0, +2]":<20} {"Dirac touch":<20}')
print(f'{"Honeycomb":<12} {"[-3, +3]":<20} {"[-1, +1]":<20} {"[0, 0]":<20}')
print(f'{"Triangular":<12} {"[+6]":<20} {"[-2]":<20} {"[0]":<20}')

### What happens when we add atoms between kagome layers?

In real materials, kagome layers are never isolated — there are always atoms between
the layers (filler/interstitial sites). These interstitial atoms hybridize with the
kagome orbitals and can **destroy the flat band** by giving it dispersion.

Below we model this by adding an orbital at the cell center coupled to the kagome
sites with hopping strength t'. As t' increases, the flat band broadens.

In [ ]:
# Kagome + interstitial atom: effect of interlayer coupling t'
# Interstitial at (0, 0) has 4 NN kagome sites at distance a/2
t_prime_values = [0.0, 0.2, 0.5, 1.0]

fig, axes = plt.subplots(1, len(t_prime_values), figsize=(16, 4.5), sharey=True)

for ax, tp in zip(axes, t_prime_values):
    orb_4 = [[0.5, 0.0], [0.0, 0.5], [0.5, 0.5], [0.0, 0.0]]
    model = make_model(lat_vecs, orb_4)

    # Kagome NN hoppings (t = +1)
    model.set_hop(t, 0, 1, [0, 0])
    model.set_hop(t, 0, 2, [0, 0])
    model.set_hop(t, 1, 2, [0, 0])
    model.set_hop(t, 0, 2, [0, -1])
    model.set_hop(t, 0, 1, [1, -1])
    model.set_hop(t, 1, 2, [-1, 0])

    # Interstitial <-> kagome NN hoppings (4 bonds at distance a/2)
    if tp != 0:
        model.set_hop(tp, 3, 0, [0, 0])
        model.set_hop(tp, 3, 1, [0, 0])
        model.set_hop(tp, 3, 0, [-1, 0])
        model.set_hop(tp, 3, 1, [0, -1])

    k_vec, k_dist, k_node = model.k_path(k_nodes, nk, report=False)
    evals = model.solve_ham(k_vec)

    for band_idx in range(evals.shape[1]):
        ax.plot(k_dist, evals[:, band_idx], color='#9C7AC7', linewidth=1.8)
    # Mark flat band reference
    ax.axhline(-2, color='red', linewidth=0.6, linestyle=':', alpha=0.4)
    for node in k_node:
        ax.axvline(node, color='gray', linewidth=0.5, linestyle='--')
    ax.set_xticks(k_node)
    ax.set_xticklabels(k_labels)
    ax.set_xlim(k_dist[0], k_dist[-1])
    ax.set_title(f"t' / t = {tp:.1f}", fontsize=12)
    ax.grid(True, alpha=0.1)

axes[0].set_ylabel('E / t', fontsize=12)
fig.suptitle('Kagome + interstitial: coupling to filler atoms destroys the flat band',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print("t'/t = 0.0 : Pure kagome + decoupled interstitial (flat bands at E=-2t and E=0)")
print("t'/t = 0.2 : Weak coupling — kagome flat band acquires slight dispersion")
print("t'/t = 0.5 : Moderate — flat band clearly broadened, hybridization gap opens")
print("t'/t = 1.0 : Strong — all bands strongly hybridized, flat band destroyed")

### Effect of atom type: hopping strength controls band width

Different elements on the kagome sites correspond to different hopping integrals.
3d transition metals (Mn, Fe, Co) have relatively narrow d-bands (small |t|),
while 4d/5d metals or p-orbital systems (graphene) have larger |t|.

Below we vary |t| on the pure kagome lattice. The **flat band remains perfectly flat
regardless of hopping strength** — it is a topological property of the kagome geometry,
not a fine-tuning accident. Only the bandwidth of the dispersive bands changes.

This robustness is exactly why materials scientists are so interested in finding
new kagome compounds — the flat band physics is guaranteed by geometry.

In [ ]:
# Kagome with different hopping strengths (mimicking different atom types)
# t scales all eigenvalues linearly — the flat band stays perfectly flat
hop_values = [
    (0.3, 'Mn-like (3d)\nt = 0.3 eV', '#9C7AC7'),
    (0.5, 'Fe-like (3d)\nt = 0.5 eV', '#E06633'),
    (1.0, 'Co-like (3d)\nt = 1.0 eV', '#F090A0'),
    (2.0, 'Ru-like (4d)\nt = 2.0 eV', '#248F8F'),
]

fig, axes = plt.subplots(1, len(hop_values), figsize=(16, 4.5), sharey=True)

for ax, (t_hop, label, color) in zip(axes, hop_values):
    kag_v = make_model(lat_vecs, orb_kag)
    kag_v.set_hop(t_hop, 0, 1, [0, 0])
    kag_v.set_hop(t_hop, 0, 2, [0, 0])
    kag_v.set_hop(t_hop, 1, 2, [0, 0])
    kag_v.set_hop(t_hop, 0, 2, [0, -1])
    kag_v.set_hop(t_hop, 0, 1, [1, -1])
    kag_v.set_hop(t_hop, 1, 2, [-1, 0])

    k_vec, k_dist, k_node = kag_v.k_path(k_nodes, nk, report=False)
    evals = kag_v.solve_ham(k_vec)

    for band_idx in range(evals.shape[1]):
        ax.plot(k_dist, evals[:, band_idx], color=color, linewidth=2.0)
    for node in k_node:
        ax.axvline(node, color='gray', linewidth=0.5, linestyle='--')
    ax.axhline(0, color='black', linewidth=0.3, alpha=0.3)
    ax.set_xticks(k_node)
    ax.set_xticklabels(k_labels)
    ax.set_xlim(k_dist[0], k_dist[-1])
    ax.set_title(label, fontsize=10)
    ax.grid(True, alpha=0.1)

axes[0].set_ylabel('Energy (eV)', fontsize=12)
fig.suptitle('Kagome bands scale with hopping t — flat band remains perfectly flat',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print('The flat band at E = -2t is perfectly flat for ALL values of t.')
print('This is a geometric property of the kagome lattice, not fine-tuning.')
print('Larger |t| (heavier elements / wider orbitals) → wider total bandwidth.')

---
## 1.7 Interactive 3D visualization

The matplotlib 3D plots above are fine for quick inspection, but for publication-quality
rendering we can use **PyVista** — a 3D visualization library built on VTK.

The helper module `scripts/crystal_viz.py` provides:
- Jmol-standard element colors and van-der-Waals radii
- Periodic boundary atom images (atoms at cell faces appear on both sides)
- Unit cell wireframe with supercell support
- **Interactive controls** — sliders for atom scale, camera angle, zoom, and supercell size

In [ ]:
import sys, os
PROJECT_DIR = os.environ.get('PROJECT_ROOT', '/content/APS_demo_SCIGEN')
if os.path.join(PROJECT_DIR, 'scripts') not in sys.path:
    sys.path.insert(0, os.path.join(PROJECT_DIR, 'scripts'))

# Pull latest scripts if running from a cloned repo (ensures plot_crystal_interactive exists)
if os.path.exists(os.path.join(PROJECT_DIR, '.git')):
    os.system(f'cd {PROJECT_DIR} && git pull -q 2>/dev/null || true')

try:
    from crystal_viz import plot_crystal, plot_crystal_grid
    # Try importing interactive viewer (added in later commit)
    try:
        from crystal_viz import plot_crystal_interactive
        viewer = plot_crystal_interactive(nacl, title='NaCl (rock salt)',
                                          atom_scale_default=0.3, elevation_default=60)
        display(viewer)
    except ImportError:
        # Fallback to static render if interactive not available
        img = plot_crystal(nacl, title='NaCl (rock salt)', atom_scale=0.3, elevation=60)
except ImportError as e:
    print(f'PyVista not available ({e}) — skipping 3D rendering.')
    print('Install with: pip install pyvista')
except Exception as e:
    print(f'3D rendering failed: {e}')
    print('This is expected if running without a display server.')

In [ ]:
try:
    try:
        from crystal_viz import plot_crystal_interactive
        viewer_kag = plot_crystal_interactive(kagome_struct, title='Kagome lattice (Mn)',
                                              supercell=(2, 2, 1),
                                              atom_scale_default=0.3, elevation_default=80)
        display(viewer_kag)
    except ImportError:
        from crystal_viz import plot_crystal
        img_kag = plot_crystal(kagome_struct, title='Kagome lattice (Mn)',
                               supercell=(2, 2, 1), atom_scale=0.3, elevation=80)
except Exception as e:
    print(f'Kagome rendering skipped: {e}')

---
## 1.8 Space group analysis

Every crystal structure belongs to one of 230 **space groups** — symmetry classifications
that describe the rotations, reflections, and translations that leave the structure invariant.

Pymatgen can automatically determine the space group of any structure using the
`SpacegroupAnalyzer` class.

In [ ]:
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer

# Analyze NaCl
sga_nacl = SpacegroupAnalyzer(nacl)
print('=== NaCl ===')
print(f'  Space group symbol:  {sga_nacl.get_space_group_symbol()}')
print(f'  Space group number:  {sga_nacl.get_space_group_number()}')
print(f'  Crystal system:      {sga_nacl.get_crystal_system()}')
print(f'  Point group:         {sga_nacl.get_point_group_symbol()}')
print(f'  Lattice type:        {sga_nacl.get_lattice_type()}')
print()

# Analyze the structure loaded from dataset
sga_ds = SpacegroupAnalyzer(struct)
print(f'=== {row["pretty_formula"]} ({row["material_id"]}) ===')
print(f'  Space group symbol:  {sga_ds.get_space_group_symbol()}')
print(f'  Space group number:  {sga_ds.get_space_group_number()}')
print(f'  Crystal system:      {sga_ds.get_crystal_system()}')
print(f'  Point group:         {sga_ds.get_point_group_symbol()}')
print(f'  Lattice type:        {sga_ds.get_lattice_type()}')
print()

# Analyze kagome lattice
sga_kag = SpacegroupAnalyzer(kagome_struct)
print(f'=== Kagome (Mn) ===')
print(f'  Space group symbol:  {sga_kag.get_space_group_symbol()}')
print(f'  Space group number:  {sga_kag.get_space_group_number()}')
print(f'  Crystal system:      {sga_kag.get_crystal_system()}')

In [ ]:
import matplotlib.pyplot as plt

# Space group distribution in the training data
sg_counts = df['spacegroup.number'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(sg_counts.index, sg_counts.values, width=1.0, color='steelblue', edgecolor='none')
ax.set_xlabel('Space group number', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Space group distribution in MP-20 training data', fontsize=13)
ax.set_xlim(0, 231)

# Mark the 7 crystal systems
systems = [(1, 2, 'Triclinic'), (3, 15, 'Monoclinic'), (16, 74, 'Orthorhombic'),
           (75, 142, 'Tetragonal'), (143, 167, 'Trigonal'),
           (168, 194, 'Hexagonal'), (195, 230, 'Cubic')]
colors = ['#e6194b', '#f58231', '#ffe119', '#3cb44b', '#42d4f4', '#4363d8', '#911eb4']
for (lo, hi, name), c in zip(systems, colors):
    ax.axvspan(lo - 0.5, hi + 0.5, alpha=0.08, color=c)
    ax.text((lo + hi) / 2, ax.get_ylim()[1] * 0.92, name,
            ha='center', fontsize=7, color=c, fontweight='bold')

plt.tight_layout()
plt.show()

---
## 1.9 Simulated X-ray diffraction (XRD)

X-ray diffraction is the primary experimental technique for determining crystal structures.
Pymatgen can simulate the **powder XRD pattern** from any structure, which is useful for:
- Comparing generated structures against experimental data
- Identifying the crystal phase of a material
- Validating that a generated structure matches the intended symmetry

In [ ]:
from pymatgen.analysis.diffraction.xrd import XRDCalculator

xrd_calc = XRDCalculator(wavelength='CuKa')  # Cu Kα radiation (1.5406 Å)

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

# NaCl XRD pattern
pattern_nacl = xrd_calc.get_pattern(nacl)
axes[0].stem(pattern_nacl.x, pattern_nacl.y, linefmt='steelblue', markerfmt=' ', basefmt=' ')
axes[0].set_ylabel('Intensity (%)', fontsize=11)
axes[0].set_title('Simulated XRD: NaCl', fontsize=12)
axes[0].set_xlim(10, 90)
# Label the strongest peaks
for x, y, hkl in zip(pattern_nacl.x, pattern_nacl.y, pattern_nacl.hkls):
    if y > 20:
        label = ''.join(str(i) for i in hkl[0]['hkl'])
        axes[0].annotate(f'({label})', xy=(x, y), xytext=(x, y + 5),
                         fontsize=7, ha='center', color='steelblue')

# Dataset structure XRD pattern
pattern_ds = xrd_calc.get_pattern(struct)
axes[1].stem(pattern_ds.x, pattern_ds.y, linefmt='coral', markerfmt=' ', basefmt=' ')
axes[1].set_xlabel('2θ (degrees)', fontsize=11)
axes[1].set_ylabel('Intensity (%)', fontsize=11)
axes[1].set_title(f'Simulated XRD: {row["pretty_formula"]}', fontsize=12)
axes[1].set_xlim(10, 90)

plt.tight_layout()
plt.show()

print(f'NaCl: {len(pattern_nacl.x)} reflections')
print(f'{row["pretty_formula"]}: {len(pattern_ds.x)} reflections')

---
### Key takeaways: why we need generative models

1. **Materials databases are large** (~150K structures in MP) but they mostly cover
   well-known, thermodynamically stable materials.
2. **Exotic geometries are rare** — kagome, honeycomb, and other frustrated lattices
   appear in only a handful of known structures.
3. **Generative models** can help explore the vast space of *possible* materials
   that haven't been made yet.

In the next notebook, we'll learn how these generative models work.

---
## Appendix: Live Materials Project API queries

The [Materials Project](https://materialsproject.org) (~150,000 computed structures)
can be queried directly with the `mp-api` Python client. This section is **optional** —
all data needed for the tutorial is already loaded above.

> Get a free API key at [materialsproject.org/api](https://materialsproject.org/api)

In [ ]:
# Set your API key here (or leave as None to use fallback data)
MP_API_KEY = None  # e.g., 'your_api_key_here'

USE_MP_API = MP_API_KEY is not None

if USE_MP_API:
    from mp_api.client import MPRester
    mpr = MPRester(MP_API_KEY)
    print('MP API client ready.')
else:
    print('No API key provided — using pre-downloaded data.')
    print('(To enable live queries, get a free key at materialsproject.org/api)')

### Querying for Mn-containing materials

In [ ]:
if USE_MP_API:
    # Search for stable Mn-containing materials
    results = mpr.materials.summary.search(
        elements=['Mn'],
        energy_above_hull=(0, 0.05),  # near-stable materials
        num_sites=(1, 20),  # small unit cells (like SCIGEN's training data)
        fields=['material_id', 'formula_pretty', 'formation_energy_per_atom',
                'energy_above_hull', 'nsites', 'symmetry']
    )
    print(f'Found {len(results)} near-stable Mn-containing materials with ≤20 atoms/cell')
    print()
    for r in results[:10]:
        print(f'  {r.material_id}  {r.formula_pretty:<15s}  '
              f'Ef={r.formation_energy_per_atom:.3f} eV/atom  '
              f'Ehull={r.energy_above_hull:.3f} eV  '
              f'nsites={r.nsites}')
else:
    print('(Skipping live query — no API key)')

In [ ]:
if USE_MP_API:
    # Download a specific structure
    struct_mp = mpr.get_structure_by_material_id('mp-35')  # Mn metal
    print(f'Downloaded: Mn metal (mp-35)')
    print(struct_mp)
else:
    print('(Skipping live download — no API key)')

---
## References

- **pymatgen:** Ong et al., "Python Materials Genomics (pymatgen): A robust, open-source python library for materials analysis," *Comput. Mater. Sci.* 68, 314–319 (2013). [GitHub](https://github.com/materialsproject/pymatgen)
- **PythTB:** Coh & Vanderbilt, "Python Tight Binding," [pythtb.org](http://www.physics.rutgers.edu/pythtb/)
- **PyVista:** Sullivan & Kaszynski, "PyVista: 3D plotting and mesh analysis through a streamlined interface for the Visualization Toolkit (VTK)," *JOSS* 4, 1450 (2019). [GitHub](https://github.com/pyvista/pyvista)
- **mp-api:** Rosen et al., "MP API — The Materials Project API," [GitHub](https://github.com/materialsproject/api)
- **Materials Project (MP-20 dataset):** Jain et al., "Commentary: The Materials Project," *APL Materials* 1, 011002 (2013). [materialsproject.org](https://materialsproject.org)
- **SCIGEN:** Okabe et al., "Structural constraint integration in a generative model for the discovery of quantum materials," *Nature Materials* (2025). [DOI](https://doi.org/10.1038/s41563-025-02355-y)

---
## What's next?

In **Notebook 02**, we'll learn how **generative diffusion models** work and why they're well-suited for crystal structure generation.